# **Cross-validation & Grid-search**

---
## Imports

In [1]:
# cross val & grid-search
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# modèles
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier

# python lib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import time

In [2]:
# lib pour importer le dataset
from ucimlrepo import fetch_ucirepo
# fetch dataset
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 
# data (as pandas dataframes) 
X = breast_cancer_wisconsin_diagnostic.data.features 
y = breast_cancer_wisconsin_diagnostic.data.targets 

---
## Informations sur le dataset

In [3]:
breast_cancer_wisconsin_diagnostic.metadata

{'uci_id': 17,
 'name': 'Breast Cancer Wisconsin (Diagnostic)',
 'repository_url': 'https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic',
 'data_url': 'https://archive.ics.uci.edu/static/public/17/data.csv',
 'abstract': 'Diagnostic Wisconsin Breast Cancer Database.',
 'area': 'Health and Medicine',
 'tasks': ['Classification'],
 'characteristics': ['Multivariate'],
 'num_instances': 569,
 'num_features': 30,
 'feature_types': ['Real'],
 'demographics': [],
 'target_col': ['Diagnosis'],
 'index_col': ['ID'],
 'has_missing_values': 'no',
 'missing_values_symbol': None,
 'year_of_dataset_creation': 1993,
 'last_updated': 'Fri Nov 03 2023',
 'dataset_doi': '10.24432/C5DW2B',
 'creators': ['William Wolberg',
  'Olvi Mangasarian',
  'Nick Street',
  'W. Street'],
 'intro_paper': {'ID': 230,
  'type': 'NATIVE',
  'title': 'Nuclear feature extraction for breast tumor diagnosis',
  'authors': 'W. Street, W. Wolberg, O. Mangasarian',
  'venue': 'Electronic imaging',
  'yea

In [4]:
print(breast_cancer_wisconsin_diagnostic.variables) 

                  name     role         type demographic description units  \
0                   ID       ID  Categorical        None        None  None   
1            Diagnosis   Target  Categorical        None        None  None   
2              radius1  Feature   Continuous        None        None  None   
3             texture1  Feature   Continuous        None        None  None   
4           perimeter1  Feature   Continuous        None        None  None   
5                area1  Feature   Continuous        None        None  None   
6          smoothness1  Feature   Continuous        None        None  None   
7         compactness1  Feature   Continuous        None        None  None   
8           concavity1  Feature   Continuous        None        None  None   
9      concave_points1  Feature   Continuous        None        None  None   
10           symmetry1  Feature   Continuous        None        None  None   
11  fractal_dimension1  Feature   Continuous        None        

In [5]:
print(X.shape, y.shape)

(569, 30) (569, 1)


In [6]:
X.sample(4)

,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,symmetry1,fractal_dimension1,...,radius3,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3
55,11.52,18.75,73.34,409.0,0.09524,0.05473,0.03036,0.02278,0.1920,0.05907,...,12.84,22.47,81.81,506.2,0.1249,0.08720,0.09076,0.06316,0.3306,0.07036
113,10.51,20.19,68.64,334.2,0.11220,0.13030,0.06476,0.03068,0.1922,0.07782,...,11.16,22.75,72.62,374.4,0.1300,0.20490,0.12950,0.06136,0.2383,0.09026
371,15.19,13.21,97.65,711.8,0.07963,0.06934,0.03393,0.02657,0.1721,0.05544,...,16.20,15.73,104.50,819.1,0.1126,0.17370,0.13620,0.08178,0.2487,0.06766
67,11.31,19.04,71.80,394.1,0.08139,0.04701,0.03709,0.02230,0.1516,0.05667,...,12.33,23.84,78.00,466.7,0.1290,0.09148,0.14440,0.06961,0.2400,0.06641


In [7]:
y.sample(4)

,Diagnosis
271,B
495,B
111,B
179,B


In [8]:
y.value_counts()
# Valeur binaire catégorielle B/M

Diagnosis
B            357
M            212
Name: count, dtype: int64

---
## Cross Validation

In [9]:
# Cross-validation demande un array de dimension 1d
y = np.ravel(y)
models = [KNeighborsClassifier(), svm.SVC(), SGDClassifier(), MLPClassifier(max_iter=300)]

for model in models:
    print(f"{model} {cross_val_score(model, X, y, cv = 5 ).mean():.2%}")

KNeighborsClassifier() 92.79%
SVC() 91.22%
SGDClassifier() 85.77%
MLPClassifier(max_iter=300) 92.27%


---
## Grid Search

In [10]:
y = np.ravel(y)
models = [KNeighborsClassifier(), svm.SVC(), SGDClassifier(), MLPClassifier()]

In [11]:
# Définir les grilles de recherche pour chaque modèle
KNN = {'n_neighbors': [1,2,3,4,5,6,7,8,9,10], # Grille de recherche pour KNeighborsClassifier. On va tester des valeurs de voisins de 3, 5, 7 et 9
        'weights' : ['uniform', 'distance'], 
        'algorithm' : ['auto', 'ball_tree', 'kd_tree', 'brute'],
        'metric' : ['cityblock', 'euclidean', 'manhattan', 'minkowski']
    }
SVM = {'kernel': ['linear', 'rbf'], # Grille de recherche pour SVC (noyau et paramètre C)
        'C': [0.1, 1, 10, 100], 
        'decision_function_shape': ['ovo', 'ovr']
    }
SGDC =  {'loss': ['hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive' ], 
         'penalty': ['l2', 'l1', 'elasticnet', None],
         'fit_intercept': [False, True]
    }
MLPC = {'hidden_layer_sizes': [100, 200],
        'activation': ['identity', 'logistic', 'tanh', 'relu'],
        'solver': ['sgd', 'adam'],
        'max_iter' : [1000, 1100, 1200, 1300, 1400, 1500],
    }

param_grids = [KNN, SVM, SGDC, MLPC]

In [12]:
# Création et exécution d'un GridSearchCV pour chaque modèle dans une boucle
# On va parcourir chaque model de la liste de modele et y associer chaque valeur distinct des param_grids
for model, param_grid in zip(models, param_grids):
    start = time.time()
    gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5)
    gs.fit(X, y)  # Entraînement du modèle sur les données d'entraînement (X, y)
    end = time.time()
    length = end - start
    # Pour chaque modele on affiche le meilleur score obtenu ainsi que les paramètres qui donnent ce score
    print(f"{length:2f} seconds!") # Afficher le temps de compute de chaque modèle
    print(f"Meilleur score pour {model.__class__.__name__}: {gs.best_score_:.3f}")  # Afficher le meilleur score
    print(f"Meilleurs paramètres pour {model.__class__.__name__}: {gs.best_params_}")  # Afficher les meilleurs paramètres
    print('=================================')

7.129028 seconds!
Meilleur score pour KNeighborsClassifier: 0.939
Meilleurs paramètres pour KNeighborsClassifier: {'algorithm': 'auto', 'metric': 'cityblock', 'n_neighbors': 9, 'weights': 'uniform'}
85.977793 seconds!
Meilleur score pour SVC: 0.953
Meilleurs paramètres pour SVC: {'C': 100, 'decision_function_shape': 'ovo', 'kernel': 'linear'}
2.887139 seconds!
Meilleur score pour SGDClassifier: 0.923
Meilleurs paramètres pour SGDClassifier: {'fit_intercept': False, 'loss': 'perceptron', 'penalty': None}
64.604389 seconds!
Meilleur score pour MLPClassifier: 0.944
Meilleurs paramètres pour MLPClassifier: {'activation': 'relu', 'hidden_layer_sizes': 200, 'max_iter': 1500, 'solver': 'adam'}


---
`MLPClassifier` et `SVC` obtiennent les meilleurs scores, 
Cependant ce sont les plus lents et `MLPClassifier` est un modèle stochastique avec un score qui varie à chaque compute du modèle.

`SGDClassifier` obtient le moins bon score mais il est clairement le plus rapide